# 12. Looking ahead: inspecting a dataset (milestone 2): solutions

Exercises of [chapter 12](../12_inspecting.md): a preview of milestone 2, made only from what the catalogue holds. Run the setup cell first.

These are the solutions; try the exercise notebook first.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import json
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pystac, rasterio
from shapely.geometry import shape
from data_acquisition.catalog import build, root, stac
from data_acquisition.catalog import summary as summaries
from data_acquisition.steps import curate


def product(item):
    """The product of an item: the key of its data asset."""
    return next(k for k, a in item.assets.items() if "data" in (a.roles or []))

**1. A table of items.** Build, curate and root the sample, open `<sample root>/stac/catalog.json` with pystac, and make one table row per item: id, product, kind, site, start and end of its time, date precision, number of flags.

In [ ]:
s = summaries.load(str(ROOT / "docs/manual/solutions/training_pond.yaml"))
catalog, _ = build.build(SAMPLE, s)
for item in catalog.get_items(recursive=True):
    curate.curate_item(item, s)
stac.save(catalog, SAMPLE / "stac/training_pond")
root.build_root(SAMPLE / "stac")

items = list(pystac.Catalog.from_file(str(SAMPLE / "stac/catalog.json")).get_items(recursive=True))
rows = []
for item in items:
    p = item.properties
    start = item.datetime or pystac.utils.str_to_datetime(p["start_datetime"])
    end = item.datetime or pystac.utils.str_to_datetime(p["end_datetime"])
    rows.append({"id": item.id, "product": product(item), "kind": p["beaver:kind"], "site": p["beaver:site"],
                 "start": start, "end": end, "precision": p.get("beaver:date_precision"),
                 "flags": len(p["beaver:quality"]), "geometry": shape(item.geometry)})
table = pd.DataFrame(rows)
table.drop(columns="geometry")

**2. Coverage.** Turn the table into a GeoDataFrame and plot the footprints in the site's UTM zone, coloured by product. What does each footprint cover, and why is the photos' one invisible?

In [ ]:
frame = gpd.GeoDataFrame(table, crs="EPSG:4326").to_crs("EPSG:32612")
ax = frame.plot(column="product", alpha=0.3, edgecolor="black", legend=True, figsize=(6, 6))
ax.set_title("Training Pond: item footprints (UTM 12N)")
plt.show()
print(frame.assign(area_m2=frame.area.round())[["product", "area_m2"]])
# The rasters cover their grid (20 m x 20 m), the annotations the extent of their features. The photos' footprint
# is the outline of the camera positions, not the ground the photos show: these six were taken along one line, so
# it is a sliver of almost no area (finding 17 in docs/dataset/course_findings.md).

**3. Timeline.** One row per product, a mark per item from its start to its end. Which item is an interval, and why?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
products = sorted(table["product"].unique())
for _, r in table.iterrows():
    y = products.index(r["product"])
    ax.plot([r["start"], r["end"]], [y, y], lw=8, solid_capstyle="butt")
    ax.plot([r["start"]], [y], marker="|", markersize=14, color="black")
ax.set_yticks(range(len(products)), products)
plt.show()
print(table.loc[table["start"] != table["end"], ["id", "start", "end", "precision"]])
# The DTM: the summary knows it only to the month (August 2022), so it covers the whole month.

**4. Quick-look.** Without a stored quick-look, read the orthomosaic at about 200 pixels wide (`out_shape`) and show it next to the item's flags.

In [ ]:
item = next(i for i in items if product(i) == "orthomosaic")
with rasterio.open(item.assets["orthomosaic"].get_absolute_href()) as r:
    width = 200
    height = round(r.height * width / r.width)
    rgb = r.read([1, 2, 3], out_shape=(3, height, width))
plt.imshow(np.moveaxis(rgb, 0, -1))
plt.title(item.id)
plt.axis("off")
plt.show()
print(f"read {rgb.nbytes:,d} bytes instead of {r.width * r.height * 3:,d}")
print("flags:", item.properties["beaver:quality"] or "none")

**5. What is missing.** By code: which items have a `thumbnail` asset, which have colonies, and what does `sites.json` say is still to decide?

In [ ]:
def has_thumbnail(item):
    return any("thumbnail" in (a.roles or []) for a in item.assets.values())


print("thumbnails:", [i.id for i in items if has_thumbnail(i)])
print("with colonies:", [i.id for i in items if i.properties.get("beaver:colonies")])
sites = json.loads((SAMPLE / "stac/sites.json").read_text())
for site, entry in sites.items():
    print(site, {k: entry[k] for k in ("colonies", "land_status", "governance")})
# No quick-looks and no colonies: the browser must read files at a reduced size, and the colony views wait for the
# registry's decisions (and define_aoi, milestone 5).